## Tumor Subtype Classification using MOFA factors

In [ ]:
import pandas as pd
import mofax as mfx

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score, precision_score, recall_score, f1_score, average_precision_score

In [41]:
def load_mofa_factors(model_path, clinical_index):
    model = mfx.mofa_model(model_path)
    factors = model.get_factors(df = True)
    factors = factors.loc[clinical_index]
    return factors

In [42]:
mofa_models = {
    "mofa_trained_lg2": "../../data/latent/mofa_trained_lg2.hdf5",
    "mofa_trained_vsn": "../../data/latent/mofa_trained_vsn.hdf5",
    "mofa_trained_lg2_fs": "../../data/latent/mofa_trained_lg2_fs.hdf5",
    "mofa_trained_vsn_fs": "../../data/latent/mofa_trained_vsn_fs.hdf5"
}

In [43]:
clinical_data = pd.read_csv('../../data/cleaned_data/clinical_cleaned.csv', index_col = 0)
y = clinical_data['tumor_subtype'].map({'other_subtype': 1, 'ductal_type': 0})

In [44]:
X = load_mofa_factors(mofa_models['mofa_trained_lg2'], clinical_data.index)

#### Training

In [45]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4, 5],
    'min_samples_leaf': [5, 10, 15, 20],
    'class_weight': ['balanced', 'balanced_subsample']
}

In [46]:
scoring = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision': make_scorer(precision_score, zero_division = 0),
    'recall': make_scorer(recall_score, zero_division = 0),
    'f1': make_scorer(f1_score, zero_division = 0),
    'pr_auc_score': make_scorer(average_precision_score, response_method='predict_proba'),
}

In [47]:
outer_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
inner_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [31]:
rows_rf = []

for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf = RandomForestClassifier(random_state=42, n_jobs=-1)

    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid_rf,
        scoring=scoring,
        refit="pr_auc_score",
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True,
    )

    grid_search.fit(X_train, y_train)

    best_index = grid_search.best_index_
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]

    row = {
        "fold": fold_id,
        "best_params": grid_search.best_params_,
    }

    for metric in scoring.keys():
        row[f"inner_train_{metric}"] = grid_search.cv_results_[f"mean_train_{metric}"][best_index]
        row[f"inner_validation_{metric}"] = grid_search.cv_results_[f"mean_test_{metric}"][best_index]

    row["outer_test_balanced_accuracy"] = balanced_accuracy_score(y_test, y_pred)
    row["outer_test_precision"] = precision_score(y_test, y_pred, zero_division=0)
    row["outer_test_recall"] = recall_score(y_test, y_pred, zero_division=0)
    row["outer_test_f1"] = f1_score(y_test, y_pred, zero_division=0)
    row["outer_test_pr_auc"] = average_precision_score(y_test, y_prob)

    rows_rf.append(row)

In [32]:
nested_results_df_rf = pd.DataFrame(rows_rf)
display(nested_results_df_rf)

,fold,best_params,inner_train_balanced_accuracy,inner_validation_balanced_accuracy,inner_train_precision,inner_validation_precision,inner_train_recall,inner_validation_recall,inner_train_f1,inner_validation_f1,inner_train_pr_auc_score,inner_validation_pr_auc_score,outer_test_balanced_accuracy,outer_test_precision,outer_test_recall,outer_test_f1,outer_test_pr_auc
0,1,"{'class_weight': 'balanced_subsample', 'max_de...",0.813268,0.647971,0.895421,0.700000,0.641520,0.33,0.747253,0.413333,0.905586,0.510749,0.750000,1.000000,0.500000,0.666667,0.714646
1,2,"{'class_weight': 'balanced', 'max_depth': 2, '...",0.753350,0.669457,0.808485,0.646667,0.532164,0.39,0.638822,0.475238,0.805973,0.497919,0.750000,1.000000,0.500000,0.666667,0.593162
2,3,"{'class_weight': 'balanced', 'max_depth': 2, '...",0.800671,0.628623,0.804322,0.666667,0.630994,0.30,0.706572,0.397460,0.813619,0.476969,0.698276,0.500000,0.500000,0.500000,0.584827
3,4,"{'class_weight': 'balanced_subsample', 'max_de...",0.746296,0.698804,0.775175,0.700000,0.522222,0.44,0.623065,0.533333,0.793942,0.555494,0.583333,1.000000,0.166667,0.285714,0.429060
4,5,"{'class_weight': 'balanced', 'max_depth': 4, '...",0.894121,0.701667,0.917386,0.866667,0.803158,0.42,0.855195,0.545238,0.955427,0.562638,0.566667,0.333333,0.200000,0.250000,0.403152


That code gives the mean across the 5 outer folds for each metric.

So for each metric:

- Train = mean of the selected model’s inner training score across outer folds

- Validation = mean of the selected model’s inner validation score across outer folds

- Test = mean of the outer test score across outer folds

In [33]:
summary_rows_rf = []

metrics = ["balanced_accuracy", "precision", "recall", "f1", "pr_auc"]

for metric in metrics:
    inner_metric = "pr_auc_score" if metric == "pr_auc" else metric
    summary_rows_rf.append({
        "metric": metric,
        "Train": nested_results_df_rf[f"inner_train_{inner_metric}"].mean(),
        "Validation": nested_results_df_rf[f"inner_validation_{inner_metric}"].mean(),
        "Test": nested_results_df_rf[f"outer_test_{metric}"].mean(),
    })

comparison_df_rf = pd.DataFrame(summary_rows_rf).set_index("metric")
display(comparison_df_rf)


,Train,Validation,Test
metric,,,
balanced_accuracy,0.801541,0.669304,0.669655
precision,0.840158,0.716000,0.766667
recall,0.626012,0.376000,0.373333
f1,0.714181,0.472921,0.473810
pr_auc,0.854909,0.520754,0.544970


SVM

In [48]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(class_weight='balanced', dual = "auto", random_state=42, max_iter=10000)),])

In [49]:
param_grid_svm = {'svm__C': [0.01, 0.1, 1, 10, 100],}

In [50]:
scoring = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'precision': make_scorer(precision_score, zero_division = 0),
    'recall': make_scorer(recall_score, zero_division = 0),
    'f1': make_scorer(f1_score, zero_division = 0),
    'pr_auc_score': make_scorer(average_precision_score, response_method='decision_function'),
}

outer_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
inner_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [51]:
rows_svm = []

for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid_svm,
        scoring=scoring,
        refit="pr_auc_score",
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True,
    )

    grid_search.fit(X_train, y_train)

    best_index = grid_search.best_index_
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    y_score = best_model.decision_function(X_test)

    row = {
        "fold": fold_id,
        "best_params": grid_search.best_params_,
        "inner_train_balanced_accuracy": grid_search.cv_results_[f"mean_train_balanced_accuracy"][best_index],
        "inner_validation_balanced_accuracy": grid_search.cv_results_[f"mean_test_balanced_accuracy"][best_index],
        "outer_test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "inner_train_precision": grid_search.cv_results_[f"mean_train_precision"][best_index],
        "inner_validation_precision": grid_search.cv_results_[f"mean_test_precision"][best_index],
        "outer_test_precision": precision_score(y_test, y_pred, zero_division=0),
        "inner_train_recall": grid_search.cv_results_[f"mean_train_recall"][best_index],
        "inner_validation_recall": grid_search.cv_results_[f"mean_test_recall"][best_index],
        "outer_test_recall": recall_score(y_test, y_pred, zero_division=0),
        "inner_train_f1": grid_search.cv_results_[f"mean_train_f1"][best_index],
        "inner_validation_f1": grid_search.cv_results_[f"mean_test_f1"][best_index],
        "outer_test_f1": f1_score(y_test, y_pred, zero_division=0),
        "inner_train_pr_auc": grid_search.cv_results_[f"mean_train_pr_auc_score"][best_index],
        "inner_validation_pr_auc": grid_search.cv_results_[f"mean_test_pr_auc_score"][best_index],
        "outer_test_pr_auc": average_precision_score(y_test, y_score)
    }

    rows_svm.append(row)

In [52]:
nested_results_df_svm = pd.DataFrame(rows_svm)
display(nested_results_df_svm)

,fold,best_params,inner_train_balanced_accuracy,inner_validation_balanced_accuracy,outer_test_balanced_accuracy,inner_train_precision,inner_validation_precision,outer_test_precision,inner_train_recall,inner_validation_recall,outer_test_recall,inner_train_f1,inner_validation_f1,outer_test_f1,inner_train_pr_auc,inner_validation_pr_auc,outer_test_pr_auc
0,1,{'svm__C': 0.01},0.696546,0.699203,0.700000,0.414433,0.488889,0.500000,0.553216,0.56,0.5,0.471568,0.474026,0.500000,0.613570,0.563212,0.588505
1,2,{'svm__C': 0.01},0.712637,0.631413,0.698276,0.395102,0.319048,0.500000,0.607602,0.49,0.5,0.477258,0.371306,0.500000,0.583857,0.539477,0.639938
2,3,{'svm__C': 0.01},0.727385,0.564022,0.612069,0.436627,0.220159,0.272727,0.618129,0.34,0.5,0.503597,0.263077,0.352941,0.611362,0.427578,0.593169
3,4,{'svm__C': 0.01},0.725559,0.614746,0.732759,0.513972,0.353810,0.750000,0.554971,0.44,0.5,0.531751,0.369281,0.600000,0.636585,0.487727,0.539092
4,5,{'svm__C': 0.01},0.711013,0.634384,0.733333,0.436880,0.375000,0.428571,0.573684,0.43,0.6,0.495042,0.367011,0.500000,0.643503,0.524296,0.441250


In [53]:
summary_rows_svm = []

metrics = ["balanced_accuracy", "precision", "recall", "f1", "pr_auc"]

for metric in metrics:
    summary_rows_svm.append({
        "metric": metric,
        "Train": nested_results_df_svm[f"inner_train_{metric}"].mean(),
        "Validation": nested_results_df_svm[f"inner_validation_{metric}"].mean(),
        "Test": nested_results_df_svm[f"outer_test_{metric}"].mean(),
    })

comparison_df_svm = pd.DataFrame(summary_rows_svm).set_index("metric")
display(comparison_df_svm)

,Train,Validation,Test
metric,,,
balanced_accuracy,0.714628,0.628754,0.695287
precision,0.439403,0.351381,0.490260
recall,0.581520,0.452000,0.520000
f1,0.495843,0.368940,0.490588
pr_auc,0.617776,0.508458,0.560391
